# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides an end-to-end guide for loading and exploring the FAIR<sup>2</sup> dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Display main metadata fields
md = dataset.metadata
print(f"Dataset Name: {md.name}\n")
print(f"Description: {md.description}\n")
print(f"Published: {md.datePublished}")


## 2. Data Overview
Review available record sets, fields, and their `@id`s.

We print all the data tables (record sets), as well as their fields. All Croissant entities are referenced by their `@id`.


In [ ]:
# List all record sets in the dataset by @id and their fields by @id

def print_record_sets_overview(ds):
    print("Available record sets:")
    for rs in ds.metadata.record_sets:
        print(f"- RecordSet @id: {rs.id}")
        print(f"  Name: {rs.name if hasattr(rs, 'name') else ''}")
        print(f"  Description: {rs.description if hasattr(rs, 'description') else ''}")
        print(f"  Fields (by @id):")
        for field in rs.fields:
            print(f"    - {field.id}")
        print()

print_record_sets_overview(dataset)

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.


In [ ]:
# List all record set @ids
record_sets = [rs.id for rs in dataset.metadata.record_sets]
print("Record set @ids:", record_sets)

# For demonstration, pick the first record set (usually the main data table)
main_record_set_id = record_sets[0]

# Load all records from the main record set
records = list(dataset.records(record_set=main_record_set_id))
df = pd.DataFrame(records)
print(f"Loaded {len(df)} records from record set {main_record_set_id}.")
print("Columns in DataFrame:")
print(df.columns.tolist())
df.head()

## 4. Exploratory Data Analysis (EDA)
Let's look at the numeric fields for initial analysis. 

Below, we will:
- Select a numeric field (e.g. protein coverage, peptide count, or similar depending on the dataset fields).
- Filter records based on a threshold.
- Normalize the numeric field.
- Optionally group by another field (e.g. a protein class or type).

**Note**: Use only `@id` for all fields, as shown in column names in the previous output.

In [ ]:
# Find a representative numeric field by @id
numeric_fields = [col for col in df.columns if df[col].dtype in [int, float, "int64", "float64"]]
if len(numeric_fields) == 0:
    # Try to convert all columns to numeric if possible, ignore errors
    for col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="ignore")
    numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]

print("Numeric fields by @id:", numeric_fields)
# If viable, pick the first numeric field
if len(numeric_fields) > 0:
    numeric_field_id = numeric_fields[0]
    # Pick a sample threshold for filtering
    threshold = df[numeric_field_id].dropna().quantile(0.75)

    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold:.3f}:")
    print(filtered_df.head())

    # Normalize the field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    )
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Try grouping by another (categorical) field
    group_fields = [col for col in df.columns if col != numeric_field_id and df[col].nunique() < len(df) // 4]
    if len(group_fields):
        group_field = group_fields[0]
        print(f"\nGrouping by field @id: {group_field}")
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean()
        print(grouped_df.head())
else:
    print("No numeric fields found for analysis.")

## 5. Visualization
Visualize the distribution of the selected numeric field, and relationship with any available groupings.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

if len(numeric_fields) > 0:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # If group_field exists, show a boxplot
    if 'group_field' in locals():
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=group_field, y=numeric_field_id, data=filtered_df)
        plt.title(f"{numeric_field_id} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field_id)
        plt.show()

## 6. Conclusion

In this notebook, we demonstrated how to:
- Load and explore the FAIR<sup>2</sup> Croissant dataset using `mlcroissant`.
- List available record sets and fields by their Croissant `@id`s.
- Extract a records table and perform basic EDA and visualizations using only field `@id`s.

**Continue with further biological or statistical analysis tailored to your research context!**